In [1]:
import pandas as pd

In [3]:
dc = pd.read_csv("../raw/Datacenters distribution in India.csv")

In [4]:
print("=" * 70)
print("SHAPE")
print("=" * 70)
print(dc.shape)

print("\n" + "=" * 70)
print("COLUMNS")
print("=" * 70)
print(dc.columns.tolist())

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)
print(dc.dtypes)

print("\n" + "=" * 70)
print("FIRST 10 ROWS")
print("=" * 70)
print(dc.head(10))

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)
print(dc.isna().sum())

print("\n" + "=" * 70)
print("DUPLICATE ROWS")
print("=" * 70)
print("Duplicate rows:", dc.duplicated().sum())

print("\n" + "=" * 70)
print("UNIQUE VALUES PER COLUMN")
print("=" * 70)

for col in dc.columns:
    print(f"\n{col}:")
    print("  Unique:", dc[col].nunique(dropna=False))
    if dc[col].nunique(dropna=False) <= 50:
        print(dc[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("NUMERIC SUMMARY")
print("=" * 70)
print(dc.describe(include="all").T)

SHAPE
(34, 2)

COLUMNS
['Market', 'Data Centers']

DATA TYPES
Market            str
Data Centers    int64
dtype: object

FIRST 10 ROWS
        Market  Data Centers
0       Mumbai            60
1      Chennai            33
2    Hyderabad            36
3  Navi Mumbai            25
4    Bangalore            31
5        Noida            19
6         Pune            17
7    New Delhi            12
8    Ahmedabad            11
9      Kolkata             8

MISSING VALUES
Market          0
Data Centers    0
dtype: int64

DUPLICATE ROWS
Duplicate rows: 0

UNIQUE VALUES PER COLUMN

Market:
  Unique: 34
Market
Mumbai           1
Chennai          1
Hyderabad        1
Navi Mumbai      1
Bangalore        1
Noida            1
Pune             1
New Delhi        1
Ahmedabad        1
Kolkata          1
Cochin           1
Visakhapatnam    1
Jaipur           1
Gurgaon          1
Coimbatore       1
Bhubaneswar      1
Indore           1
Raipur           1
Lucknow          1
Nashik           1
Ludhiana    

In [5]:
dc_clean = dc.rename(columns={
    "Market": "market",
    "Data Centers": "data_center_count"
}).copy()

print(dc_clean.head())
print("\nShape:", dc_clean.shape)
print("\nMissing values:")
print(dc_clean.isna().sum())

        market  data_center_count
0       Mumbai                 60
1      Chennai                 33
2    Hyderabad                 36
3  Navi Mumbai                 25
4    Bangalore                 31

Shape: (34, 2)

Missing values:
market               0
data_center_count    0
dtype: int64


In [6]:
market_state_map = {
    "Mumbai": "Maharashtra",
    "Chennai": "Tamil Nadu",
    "Hyderabad": "Telangana",
    "Navi Mumbai": "Maharashtra",
    "Bangalore": "Karnataka",
    "Noida": "Uttar Pradesh",
    "Pune": "Maharashtra",
    "New Delhi": "Delhi",
    "Ahmedabad": "Gujarat",
    "Kolkata": "West Bengal",
    "Cochin": "Kerala",
    "Visakhapatnam": "Andhra Pradesh",
    "Jaipur": "Rajasthan",
    "Gurgaon": "Haryana",
    "Coimbatore": "Tamil Nadu",
    "Bhubaneswar": "Odisha",
    "Indore": "Madhya Pradesh",
    "Raipur": "Chhattisgarh",
    "Lucknow": "Uttar Pradesh",
    "Nashik": "Maharashtra",
    "Ludhiana": "Punjab",
    "Rohtak": "Haryana",
    "Jamnagar": "Gujarat",
    "Panchkula": "Haryana",
    "Agartala": "Tripura",
    "Guntur": "Andhra Pradesh",
    "Mohali": "Punjab",
    "Alwar": "Rajasthan",
    "Madurai": "Tamil Nadu",
    "Pondicherry": "Puducherry",
    "Patna": "Bihar",
    "Bhopal": "Madhya Pradesh",
    "Faridabad": "Haryana",
    "Murshidabad": "West Bengal"
}

dc_clean["state_name"] = dc_clean["market"].map(market_state_map)

print("Missing mappings:", dc_clean["state_name"].isna().sum())

print("\nUnmapped markets:")
print(dc_clean.loc[dc_clean["state_name"].isna(), "market"].tolist())

print("\nMarket → State:")
print(dc_clean[["market", "state_name", "data_center_count"]].to_string(index=False))

Missing mappings: 0

Unmapped markets:
[]

Market → State:
       market     state_name  data_center_count
       Mumbai    Maharashtra                 60
      Chennai     Tamil Nadu                 33
    Hyderabad      Telangana                 36
  Navi Mumbai    Maharashtra                 25
    Bangalore      Karnataka                 31
        Noida  Uttar Pradesh                 19
         Pune    Maharashtra                 17
    New Delhi          Delhi                 12
    Ahmedabad        Gujarat                 11
      Kolkata    West Bengal                  8
       Cochin         Kerala                  7
Visakhapatnam Andhra Pradesh                  6
       Jaipur      Rajasthan                  5
      Gurgaon        Haryana                  5
   Coimbatore     Tamil Nadu                  3
  Bhubaneswar         Odisha                  3
       Indore Madhya Pradesh                  3
       Raipur   Chhattisgarh                  2
      Lucknow  Uttar Pradesh 

In [9]:
dim_state = pd.read_csv("../clean/dim_state.csv")

In [10]:

dc_clean = dc_clean.merge(
    dim_state[["state_id", "state_name"]],
    on="state_name",
    how="left",
    validate="many_to_one"
)

print("Shape:", dc_clean.shape)

print("\nMissing state IDs:")
print(dc_clean["state_id"].isna().sum())

print("\nUnique markets:", dc_clean["market"].nunique())
print("Unique states:", dc_clean["state_id"].nunique())

print("\nState mapping:")
print(
    dc_clean[
        ["market", "state_id", "state_name", "data_center_count"]
    ].sort_values(["state_name", "market"]).to_string(index=False)
)

Shape: (34, 4)

Missing state IDs:
0

Unique markets: 34
Unique states: 19

State mapping:
       market state_id     state_name  data_center_count
       Guntur    IN-AP Andhra Pradesh                  1
Visakhapatnam    IN-AP Andhra Pradesh                  6
        Patna    IN-BR          Bihar                  1
       Raipur    IN-CT   Chhattisgarh                  2
    New Delhi    IN-DL          Delhi                 12
    Ahmedabad    IN-GJ        Gujarat                 11
     Jamnagar    IN-GJ        Gujarat                  1
    Faridabad    IN-HR        Haryana                  1
      Gurgaon    IN-HR        Haryana                  5
    Panchkula    IN-HR        Haryana                  1
       Rohtak    IN-HR        Haryana                  1
    Bangalore    IN-KA      Karnataka                 31
       Cochin    IN-KL         Kerala                  7
       Bhopal    IN-MP Madhya Pradesh                  1
       Indore    IN-MP Madhya Pradesh                 

In [12]:
dc_clean = dc_clean[
    [
        "market",
        "state_id",
        "state_name",
        "data_center_count"
    ]
]

dc_clean.to_csv(
    "../clean/fact_market_datacentres.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully.")
print("Shape:", dc_clean.shape)
print(dc_clean.head())

Saved successfully.
Shape: (34, 4)
        market state_id   state_name  data_center_count
0       Mumbai    IN-MH  Maharashtra                 60
1      Chennai    IN-TN   Tamil Nadu                 33
2    Hyderabad    IN-TG    Telangana                 36
3  Navi Mumbai    IN-MH  Maharashtra                 25
4    Bangalore    IN-KA    Karnataka                 31
